## ForgettingFairly — Pilot
**Hypothesis:** Continual learning forgets Task 1 unevenly across demographic subgroups.

> Runtime ~25 min on T4. Make sure GPU is enabled in notebook settings.

### Cell 3 — Set up data paths
HAM10000 has images split across two folders. We create a single flat symlink directory to avoid copying ~2GB of files.

In [ ]:
import os

HAM_ROOT = "/kaggle/input/skin-lesion-analysis-toward-melanoma-detection"
IMG_DIR  = "./data/HAM10000_images"
META_SRC = os.path.join(HAM_ROOT, "HAM10000_metadata.csv")
META_DST = "./data/HAM10000_metadata.csv"

# ── Metadata ──────────────────────────────────────────────────
if not os.path.exists(META_DST):
    import shutil
    shutil.copy(META_SRC, META_DST)
    print(f"✅ metadata copied → {META_DST}")
else:
    print(f"✅ metadata already at {META_DST}")

# ── Images: symlink both parts into one flat directory ────────
os.makedirs(IMG_DIR, exist_ok=True)
n_linked = 0

for part in ["HAM10000_images_part1", "HAM10000_images_part2"]:
    part_path = os.path.join(HAM_ROOT, part)
    if not os.path.isdir(part_path):
        print(f"⚠️  {part} not found at {part_path}")
        continue
    for fname in os.listdir(part_path):
        src = os.path.join(part_path, fname)
        dst = os.path.join(IMG_DIR, fname)
        if not os.path.exists(dst):
            os.symlink(src, dst)   # symlink = zero copy cost
            n_linked += 1

total_imgs = len(os.listdir(IMG_DIR))
print(f"✅ {n_linked} new symlinks created  |  {total_imgs} total images in {IMG_DIR}")

### Cell 4 — Sanity check the data

In [ ]:
import pandas as pd

df = pd.read_csv("./data/HAM10000_metadata.csv")
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

# HAM10000 uses 'dataset' not 'datasource' as the column name
src_col = "dataset" if "dataset" in df.columns else "datasource"
print(f"\nSource column ('{src_col}'):")
print(df[src_col].value_counts())

print("\nClass distribution (dx):")
print(df["dx"].value_counts())

print("\nSex distribution:")
print(df["sex"].value_counts(dropna=False))

# Check our two task sources are present
mel_nv = df[df["dx"].isin(["mel", "nv"])]
t1 = mel_nv[mel_nv[src_col] == "vidir_modern"]
t2 = mel_nv[mel_nv[src_col] == "rosendahl"]
print(f"\nTask 1 (vidir_modern):  {len(t1)} samples")
print(f"Task 2 (rosendahl):     {len(t2)} samples")

if len(t1) == 0 or len(t2) == 0:
    print("\n⚠️  One of the task sources is empty!")
    print("   Check the source column values above and update config.py if needed")
else:
    print("\n✅ Both task splits found – ready to run pilot")

### Cell 5 — GPU check

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram:.1f} GB")
    print("✅ GPU ready")
else:
    print("⚠️  No GPU detected – check that accelerator is set to GPU in notebook settings")
    print("   Runtime will be ~10x slower on CPU")

### Cell 6 — Run the pilot
Expected runtime: **~20–25 min on T4**.

In [ ]:
!python pilot.py \
    --epochs 10 \
    --batch_size 64 \
    --lr 1e-3 \
    --img_dir ./data/HAM10000_images \
    --metadata ./data/HAM10000_metadata.csv \
    --weights ./data/RadImageNet_resnet50.pth

### Cell 7 — Inspect results

In [ ]:
import json

with open("./results/pilot_results.json") as f:
    r = json.load(f)

m1 = r["metrics_after_task1"]
m2 = r["metrics_after_task2"]

print("=" * 55)
print("  PILOT RESULTS SUMMARY")
print("=" * 55)

print(f"\n  {'Metric':<30} {'After T1':>8}  {'After T2':>8}  {'Δ':>8}")
print(f"  {'-'*50}")

def row(label, v1, v2):
    d = v2 - v1 if (v1 is not None and v2 is not None) else None
    v1s = f"{v1:.4f}" if v1 is not None else "   N/A"
    v2s = f"{v2:.4f}" if v2 is not None else "   N/A"
    ds  = f"{d:+.4f}" if d is not None else "   N/A"
    print(f"  {label:<30} {v1s:>8}  {v2s:>8}  {ds:>8}")

row("Overall AUC",      m1["overall"]["auc"],  m2["overall"]["auc"])
row("Overall acc",      m1["overall"]["acc"],  m2["overall"]["acc"])
row("Male   AUC",       m1["male"]["auc"],     m2["male"]["auc"])
row("Female AUC",       m1["female"]["auc"],   m2["female"]["auc"])
row("Male   acc",       m1["male"]["acc"],     m2["male"]["acc"])
row("Female acc",       m1["female"]["acc"],   m2["female"]["acc"])
print(f"  {'-'*50}")
row("EOD gap |TPR_m - TPR_f|", m1["eod_gap"], m2["eod_gap"])
row("Acc gap |acc_m - acc_f|", m1["acc_gap"], m2["acc_gap"])

delta_eod = m2["eod_gap"] - m1["eod_gap"]
diff_forget = abs(
    (m2["male"]["acc"] - m1["male"]["acc"]) -
    (m2["female"]["acc"] - m1["female"]["acc"])
)

print(f"\n  Δ EOD gap:              {delta_eod:+.4f}  {'✅ worsened' if delta_eod > 0 else '⚠️ no change'}")
print(f"  Differential forgetting: {diff_forget:.4f}  {'✅ subgroups differ' if diff_forget > 0.02 else '⚠️ weak signal'}")

if delta_eod > 0 and diff_forget > 0.02:
    print("\n  🚀 HYPOTHESIS CONFIRMED – proceed to full experiment")
elif delta_eod > 0 or diff_forget > 0.02:
    print("\n  🔶 PARTIAL SIGNAL – worth investigating further")
else:
    print("\n  🔴 WEAK SIGNAL – check Cell 4 output and task splits")

### Cell 8 — Save outputs
Kaggle's `/kaggle/working/` directory persists as downloadable output after the session.

In [ ]:
import shutil, os, json

os.makedirs("/kaggle/working/outputs", exist_ok=True)

# Results JSON
shutil.copy("./results/pilot_results.json",
            "/kaggle/working/outputs/pilot_results.json")

# Checkpoint after Task 1 (useful for later experiments)
shutil.copy("./checkpoints/after_task1.pt",
            "/kaggle/working/outputs/after_task1.pt")

# Quick text summary
with open("./results/pilot_results.json") as f:
    r = json.load(f)

summary = {
    "delta_eod_gap":         r["metrics_after_task2"]["eod_gap"] - r["metrics_after_task1"]["eod_gap"],
    "delta_acc_male":        r["metrics_after_task2"]["male"]["acc"] - r["metrics_after_task1"]["male"]["acc"],
    "delta_acc_female":      r["metrics_after_task2"]["female"]["acc"] - r["metrics_after_task1"]["female"]["acc"],
    "auc_after_t1_overall":  r["metrics_after_task1"]["overall"]["auc"],
    "auc_after_t2_overall":  r["metrics_after_task2"]["overall"]["auc"],
    "config":                r["config"],
}

with open("/kaggle/working/outputs/summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved to /kaggle/working/outputs/:")
for fname in os.listdir("/kaggle/working/outputs"):
    size = os.path.getsize(f"/kaggle/working/outputs/{fname}")
    print(f"  {fname}  ({size/1e6:.1f} MB)")
print("\n✅ Download these from the Output tab on the right →")